In [ ]:
import pandas as pd

#config
nepdb_file = "../../raw/NEPdb.csv"
output_file = "../../processed/nepdb_data.csv"

valid_aa = set("ACDEFGHIKLMNPQRSTVWY")


In [31]:
## PROCESS NEPdb

with open(nepdb_file, "r") as f:
    firstline = f.readline()
sep_nep = "\t" if "\t" in firstline else ","

nep_raw = pd.read_csv(nepdb_file, sep=sep_nep, dtype=str)
nep_raw.head() 

,fpkm,id,response,Tumor Type,genesymbol,genesymbol_ref,mut_aa_pos,wt_peptide,wt_aa,mut_dna,...,num_gap,pep2aln,protein_sequence,TRAV,TRAJ,CDR3A,TRBV,TRBD,TRBJ,CDR3B
0,NaN,1,N,Colorectal cancer,C1orf159,C1orf159,106,SSGTPGRPHPGAPRVAASLFLGTFF,P,c.C317T,...,0,"""SSGTPGRPHPGAPRVAASLFLGTFF\t||||||||||||.|||||...",MALRHLALLAGLLVGVASKSMENTAQLPECCVDVVGVNASCPGASL...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,2,N,Colorectal cancer,C1orf159,C1orf159,106,SSGTPGRPHPGAPRVAASLFLGTFF,P,c.C317T,...,0,"""SSGTPGRPHPGAPRVAASLFLGTFF\t||||||||||||.|||||...",MALRHLALLAGLLVGVASKSMENTAQLPECCVDVVGVNASCPGASL...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,3,N,Colorectal cancer,MED13L,MED13L,795,VRQDNAAGRAGSSSLTQVTDLAPSL,S,c.G2384A,...,0,"""VRQDNAAGRAGSSSLTQVTDLAPSL\t||||||||||||.|||||...",MTAAANWVANGASLEDCHSNLFSLAELTGIKWRRYNFGGHGDCGPI...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,4,N,Colorectal cancer,MED13L,MED13L,795,VRQDNAAGRAGSSSLTQVTDLAPSL,S,c.G2384A,...,0,"""VRQDNAAGRAGSSSLTQVTDLAPSL\t||||||||||||.|||||...",MTAAANWVANGASLEDCHSNLFSLAELTGIKWRRYNFGGHGDCGPI...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,5,N,Endometrial cancer,KRAS,KRAS,12,VVGAGGVGK,G,NaN,...,0,"""VVGAGGVGK\t||||.||||\tVVGACGVGK""",MTEYKLVVVGAGGVGKSALTIQLIQNHFVDEYDPTIEDSYRKQVVI...,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [32]:
print(f"NEPdb: {len(nep_raw)} entries")

NEPdb: 15912 entries


In [33]:
nep_raw.columns

Index(['fpkm', 'id', 'response', 'Tumor Type', 'genesymbol', 'genesymbol_ref',
       'mut_aa_pos', 'wt_peptide', 'wt_aa', 'mut_dna', 'mut_peptide', 'mut_aa',
       'antigen_len', 'locus', 'alleleA', 'alleleB', 'APC_type',
       'antigen_type', 'minimal_peptide', 'Tcell_source', 'assay', 'sampleid',
       'PMID', 'journal', 'title', 'date', 'blockade', 'ACT', 'vaccination',
       'curative_effect', 'numpep', 'wt_pep_start', 'mut_pos_in_pep', 'num_mm',
       'num_gap', 'pep2aln', 'protein_sequence', 'TRAV', 'TRAJ', 'CDR3A',
       'TRBV', 'TRBD', 'TRBJ', 'CDR3B'],
      dtype='object')

In [34]:
#map to labels 
print(nep_raw['response'].unique())

nepdb = pd.DataFrame() 

['N' 'P']


In [35]:
#build df
nepdb["mt_seq"] = nep_raw["mut_peptide"].str.strip().str.upper()
nepdb["wt_seq"] = nep_raw["wt_peptide"].str.strip().str.upper()
nepdb["hla_allele"] = nep_raw["alleleA"].str.strip()
nepdb["mutation"] = nep_raw["mut_aa"].str.strip()
nepdb["source_molecule"] = nep_raw["genesymbol"].str.strip()
nepdb["epitope_relation"] = None
nepdb["data_source"] = "NEPdb"

nepdb.head()

,mt_seq,wt_seq,hla_allele,mutation,source_molecule,epitope_relation,data_source
0,SSGTPGRPHPGALRVAASLFLGTFF,SSGTPGRPHPGAPRVAASLFLGTFF,A*02:01,L,C1orf159,None,NEPdb
1,SSGTPGRPHPGALRVAASLFLGTFF,SSGTPGRPHPGAPRVAASLFLGTFF,A*24:01,L,C1orf159,None,NEPdb
2,VRQDNAAGRAGSNSLTQVTDLAPSL,VRQDNAAGRAGSSSLTQVTDLAPSL,A*02:01,N,MED13L,None,NEPdb
3,VRQDNAAGRAGSNSLTQVTDLAPSL,VRQDNAAGRAGSSSLTQVTDLAPSL,A*24:01,N,MED13L,None,NEPdb
4,VVGACGVGK,VVGAGGVGK,C*03:03,C,KRAS,None,NEPdb


In [36]:
nepdb["label"] = nep_raw["response"].map({"N": 0, "P": 1})
print(nepdb['label'].value_counts())

label
0    15614
1      298
Name: count, dtype: int64


In [37]:
#CLEAN MERGED
print("Before filtering invalid aa:", len(nepdb))
#drop with no pep seq
nepdb = nepdb.dropna(subset=["mt_seq"])
print(f"After dropping entries with no peptide sequence: {len(nepdb)} entries")


Before filtering invalid aa: 15912
After dropping entries with no peptide sequence: 15912 entries


In [38]:
#keep only valid aa
#filter out invalid aa
valid_aa = set("ACDEFGHIKLMNPQRSTVWY")
def is_valid(seq): 
    if not isinstance(seq, str): 
        return False
    for aa in seq:
        if aa not in valid_aa: 
            return False
    return True
mask = nepdb["mt_seq"].apply(is_valid)
nepdb = nepdb[mask]
mask2 = nepdb["wt_seq"].apply(is_valid)
nepdb = nepdb[mask2]
print("after filtering invalid aa:", len(nepdb))

after filtering invalid aa: 15912


In [39]:
#drop with no label
nepdb = nepdb.dropna(subset=["label"])
print(f"After dropping entries with no label: {len(nepdb)} entries")

After dropping entries with no label: 15912 entries


In [40]:
nepdb.head() 

,mt_seq,wt_seq,hla_allele,mutation,source_molecule,epitope_relation,data_source,label
0,SSGTPGRPHPGALRVAASLFLGTFF,SSGTPGRPHPGAPRVAASLFLGTFF,A*02:01,L,C1orf159,None,NEPdb,0
1,SSGTPGRPHPGALRVAASLFLGTFF,SSGTPGRPHPGAPRVAASLFLGTFF,A*24:01,L,C1orf159,None,NEPdb,0
2,VRQDNAAGRAGSNSLTQVTDLAPSL,VRQDNAAGRAGSSSLTQVTDLAPSL,A*02:01,N,MED13L,None,NEPdb,0
3,VRQDNAAGRAGSNSLTQVTDLAPSL,VRQDNAAGRAGSSSLTQVTDLAPSL,A*24:01,N,MED13L,None,NEPdb,0
4,VVGACGVGK,VVGAGGVGK,C*03:03,C,KRAS,None,NEPdb,0


In [41]:
#saving output file
nepdb.to_csv(output_file, index=False)